# 🏥 Early Disease Prediction — Advanced ML & Explainability Suite

### Predicting Diabetes Risk from Diagnostic Measurements

<table>
<tr>
<td><b>Dataset</b></td><td>Pima Indians Diabetes Dataset (UCI Machine Learning Repository)</td>
</tr>
<tr>
<td><b>Problem Type</b></td><td>Binary Classification (Medical Risk Prediction)</td>
</tr>
<tr>
<td><b>Target</b></td><td><code>Outcome</code> → 0 = No Diabetes, 1 = Diabetes</td>
</tr>
<tr>
<td><b>Models</b></td><td>Logistic Regression, Random Forest, SVM, XGBoost, LightGBM, Gradient Boosting, KNN, <b>Stacked Ensemble</b></td>
</tr>
<tr>
<td><b>Tuning</b></td><td>RandomizedSearchCV (5-fold, stratified)</td>
</tr>
<tr>
<td><b>Explainability</b></td><td>SHAP values, permutation importance</td>
</tr>
<tr>
<td><b>Visualization</b></td><td>Interactive Plotly dashboards + polished Matplotlib/Seaborn</td>
</tr>
</table>

---

## 📑 Table of Contents
1. [Setup & Imports](#1)
2. [Load & Inspect Data](#2)
3. [Exploratory Data Analysis](#3)
4. [Data Cleaning & Feature Engineering](#4)
5. [Train/Test Split, SMOTE & Scaling](#5)
6. [Model Training — Baseline](#6)
7. [Hyperparameter Tuning](#7)
8. [Stacked Ensemble](#8)
9. [Model Evaluation & Comparison](#9)
10. [Explainability — SHAP](#10)
11. [Interactive Risk Dashboard](#11)
12. [Live Patient Risk Predictor](#12)
13. [Conclusion & Next Steps](#13)

---

<a id="1"></a>
## 1. Setup & Imports

In [ ]:
# Install required libraries (run once)
import subprocess, sys

def pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *pkgs]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)

pip_install([
    "xgboost", "lightgbm", "imbalanced-learn", "shap", "plotly",
    "kaleido", "nbformat>=4.2.0"
])
print("✅ Libraries installed")

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

# Sklearn
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    matthews_corrcoef, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

# Boosting libraries
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Explainability
import shap

# ---- Global plotting theme ----
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

PALETTE = {
    "primary":   "#6C5CE7",
    "secondary": "#00B894",
    "danger":    "#E17055",
    "warning":   "#FDCB6E",
    "info":      "#0984E3",
    "dark":      "#2D3436",
    "light":     "#DFE6E9",
}
MODEL_COLORS = ["#6C5CE7", "#00B894", "#0984E3", "#E17055",
                 "#FDCB6E", "#A29BFE", "#FD79A8", "#00CEC9"]

PLOTLY_TEMPLATE = "plotly_white"
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
print("✅ All libraries imported · theme configured · random_state =", RANDOM_STATE)

<a id="2"></a>
## 2. Load & Inspect Data

In [ ]:
# Load Pima Indians Diabetes Dataset
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv(url)

print(f"Dataset shape   : {df.shape}")
print(f"Features        : {list(df.columns[:-1])}")
print(f"Target          : {df.columns[-1]}")
df.head(10).style.background_gradient(cmap="Purples", subset=df.columns[:-1])

In [ ]:
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info()
print()
print("Statistical Summary:")
df.describe().T.round(2).style.background_gradient(cmap="BuPu", axis=0)

<a id="3"></a>
## 3. Exploratory Data Analysis

In [ ]:
# 3.1 Interactive Target Distribution (Plotly)
counts = df["Outcome"].value_counts().sort_index()
labels = ["No Diabetes", "Diabetes"]

fig = make_subplots(
    rows=1, cols=2, specs=[[{"type": "bar"}, {"type": "domain"}]],
    subplot_titles=("Class Counts", "Class Proportion")
)
fig.add_trace(
    go.Bar(x=labels, y=counts.values, marker_color=[PALETTE["secondary"], PALETTE["danger"]],
           text=counts.values, textposition="outside", name="Count"),
    row=1, col=1
)
fig.add_trace(
    go.Pie(labels=labels, values=counts.values, hole=0.45,
           marker_colors=[PALETTE["secondary"], PALETTE["danger"]],
           textinfo="label+percent"),
    row=1, col=2
)
fig.update_layout(
    title_text="🎯 Target Variable Distribution — Outcome", title_x=0.5,
    template=PLOTLY_TEMPLATE, showlegend=False, height=420,
    font=dict(size=13)
)
fig.show()
print(f"Class imbalance ratio: {counts[0]/counts[1]:.2f} : 1  (No-Diabetes : Diabetes)")

In [ ]:
# 3.2 Missing / Zero-Value Analysis
# Zero is biologically impossible for these features -> treated as missing
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

zero_counts = {c: int((df[c] == 0).sum()) for c in zero_cols}
zero_pct = {c: round(v / len(df) * 100, 1) for c, v in zero_counts.items()}

fig = go.Figure(go.Bar(
    x=list(zero_counts.keys()), y=list(zero_counts.values()),
    text=[f"{v} ({zero_pct[k]}%)" for k, v in zero_counts.items()],
    textposition="outside",
    marker=dict(color=list(zero_counts.values()), colorscale="Reds")
))
fig.update_layout(
    title="⚠️ Zero / Missing Values per Feature", title_x=0.5,
    template=PLOTLY_TEMPLATE, height=400,
    xaxis_title="Feature", yaxis_title="Zero-value count"
)
fig.show()

In [ ]:
# 3.3 Feature Distributions by Outcome (Violin + Box, interactive)
features = df.columns[:-1].tolist()

fig = make_subplots(rows=2, cols=4, subplot_titles=features)
for i, feat in enumerate(features):
    r, c = divmod(i, 4)
    for outcome, color, name in [(0, PALETTE["secondary"], "No Diabetes"),
                                   (1, PALETTE["danger"], "Diabetes")]:
        fig.add_trace(
            go.Violin(
                y=df[df["Outcome"] == outcome][feat], name=name,
                line_color=color, side="negative" if outcome == 0 else "positive",
                points=False, showlegend=(i == 0)
            ),
            row=r + 1, col=c + 1
        )
fig.update_layout(
    title="📊 Feature Distributions by Outcome (Split Violin)", title_x=0.5,
    template=PLOTLY_TEMPLATE, height=700, violinmode="overlay"
)
fig.show()

In [ ]:
# 3.4 Correlation Heatmap (interactive)
corr = df.corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale="RdBu", zmid=0, text=corr.values, texttemplate="%{text}",
    textfont=dict(size=10), colorbar=dict(title="r")
))
fig.update_layout(
    title="🔗 Feature Correlation Matrix", title_x=0.5,
    template=PLOTLY_TEMPLATE, height=600, width=750
)
fig.show()

print("Top correlations with Outcome:")
target_corr = corr["Outcome"].drop("Outcome").abs().sort_values(ascending=False)
print(target_corr.to_string())

In [ ]:
# 3.5 3D Interactive Scatter — Glucose, BMI, Age colored by Outcome
fig = px.scatter_3d(
    df, x="Glucose", y="BMI", z="Age", color=df["Outcome"].map({0: "No Diabetes", 1: "Diabetes"}),
    color_discrete_map={"No Diabetes": PALETTE["secondary"], "Diabetes": PALETTE["danger"]},
    opacity=0.7, size_max=8,
    title="🧬 3D Risk Landscape: Glucose × BMI × Age"
)
fig.update_layout(template=PLOTLY_TEMPLATE, height=600, legend_title="Outcome")
fig.show()

In [ ]:
# 3.6 Pairplot of top correlated features (static, high quality)
top_feats = target_corr.head(5).index.tolist() + ["Outcome"]
g = sns.pairplot(
    df[top_feats], hue="Outcome", palette=[PALETTE["secondary"], PALETTE["danger"]],
    diag_kind="kde", plot_kws=dict(alpha=0.6, s=25, edgecolor="white", linewidth=0.3),
    height=2.2
)
g.fig.suptitle("Pairwise Relationships — Top 5 Predictive Features", y=1.02, fontsize=15, fontweight="bold")
plt.show()

<a id="4"></a>
## 4. Data Cleaning & Feature Engineering

In [ ]:
# 4.1 Impute biologically-impossible zeros with median
df_clean = df.copy()
for col in zero_cols:
    median_val = df_clean.loc[df_clean[col] != 0, col].median()
    df_clean[col] = df_clean[col].replace(0, median_val)
    print(f"{col:18s}: zeros -> median = {median_val:.2f}")
print("\n✅ Missing-value imputation complete")

In [ ]:
# 4.2 Feature Engineering
def bmi_category(bmi):
    if bmi < 18.5: return 0   # Underweight
    elif bmi < 25: return 1   # Normal
    elif bmi < 30: return 2   # Overweight
    else: return 3            # Obese

df_clean["BMI_Category"]     = df_clean["BMI"].apply(bmi_category)
df_clean["Age_Group"]        = pd.cut(df_clean["Age"], bins=[0, 30, 45, 60, 100],
                                       labels=[0, 1, 2, 3]).astype(int)
df_clean["Glucose_BP_Ratio"] = df_clean["Glucose"] / (df_clean["BloodPressure"] + 1)
df_clean["Insulin_Glucose"]  = df_clean["Insulin"] / (df_clean["Glucose"] + 1)
df_clean["BMI_Age"]          = df_clean["BMI"] * df_clean["Age"] / 100

new_feats = ["BMI_Category", "Age_Group", "Glucose_BP_Ratio", "Insulin_Glucose", "BMI_Age"]
print(f"New engineered features: {new_feats}")

X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]
print(f"\nFinal feature matrix : {X.shape}")
print(f"Target distribution   : {y.value_counts().to_dict()}")

<a id="5"></a>
## 5. Train/Test Split, SMOTE & Scaling

In [ ]:
# 5.1 Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# 5.2 SMOTE on training data only (avoid leakage)
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Train: {dict(pd.Series(y_train).value_counts())}")
print(f"After  SMOTE - Train: {dict(pd.Series(y_train_sm).value_counts())}")
print(f"Test set (untouched) : {dict(pd.Series(y_test).value_counts())}")

# 5.3 Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

print("\n✅ Preprocessing complete — data ready for modeling")

<a id="6"></a>
## 6. Model Training — Baseline

In [ ]:
# 6.1 Define baseline models
base_models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=1.0),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, max_depth=8),
    "SVM":                 SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE, C=1.0),
    "KNN":                 KNeighborsClassifier(n_neighbors=11),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=150, random_state=RANDOM_STATE, max_depth=3),
    "XGBoost":             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                          random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0),
    "LightGBM":            LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                           random_state=RANDOM_STATE, verbosity=-1),
}

def evaluate_model(model, X_tr, y_tr, X_te, y_te, cv_folds=5):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    cv = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(cv_folds, shuffle=True, random_state=RANDOM_STATE),
                          scoring="f1")
    return {
        "model": model, "y_pred": y_pred, "y_prob": y_prob,
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall":    recall_score(y_te, y_pred),
        "F1-Score":  f1_score(y_te, y_pred),
        "ROC-AUC":   roc_auc_score(y_te, y_prob),
        "MCC":       matthews_corrcoef(y_te, y_pred),
        "Brier":     brier_score_loss(y_te, y_prob),
        "CV Mean":   cv.mean(), "CV Std": cv.std(),
    }

print("Training baseline models...")
print("=" * 70)
results = {}
for name, model in base_models.items():
    res = evaluate_model(model, X_train_scaled, y_train_sm, X_test_scaled, y_test)
    results[name] = res
    print(f"{name:<22} F1={res['F1-Score']:.4f}  ROC-AUC={res['ROC-AUC']:.4f}  "
          f"CV={res['CV Mean']:.4f}±{res['CV Std']:.4f}")
print("=" * 70)
print("✅ Baseline training complete")

<a id="7"></a>
## 7. Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
# 7.1 Tune the top 3 baseline models by F1-score
top3 = sorted(results, key=lambda k: results[k]["F1-Score"], reverse=True)[:3]
print(f"Tuning top 3 models: {top3}")

param_grids = {
    "Random Forest": {
        "n_estimators": [100, 200, 300, 400],
        "max_depth": [4, 6, 8, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    "XGBoost": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
    },
    "LightGBM": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "num_leaves": [15, 31, 63],
    },
    "Gradient Boosting": {
        "n_estimators": [100, 150, 200],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.85, 1.0],
    },
    "SVM": {
        "C": [0.1, 1, 10, 50],
        "gamma": ["scale", "auto", 0.01, 0.1],
    },
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l2"],
        "solver": ["lbfgs", "liblinear"],
    },
    "KNN": {
        "n_neighbors": [5, 7, 9, 11, 15],
        "weights": ["uniform", "distance"],
    },
}

tuned_models = {}
for name in top3:
    base = base_models[name]
    grid = param_grids.get(name, {})
    if not grid:
        tuned_models[name] = base
        continue
    search = RandomizedSearchCV(
        base, grid, n_iter=20, scoring="f1", cv=5,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=0
    )
    search.fit(X_train_scaled, y_train_sm)
    tuned_models[name] = search.best_estimator_
    print(f"\n{name} best params: {search.best_params_}")
    print(f"{name} best CV F1  : {search.best_score_:.4f}")

print("\n✅ Hyperparameter tuning complete")

In [ ]:
# 7.2 Re-evaluate tuned models on test set
print("Tuned model performance on held-out test set:")
print("=" * 70)
for name, model in tuned_models.items():
    res = evaluate_model(model, X_train_scaled, y_train_sm, X_test_scaled, y_test)
    results[f"{name} (Tuned)"] = res
    print(f"{name+' (Tuned)':<28} F1={res['F1-Score']:.4f}  ROC-AUC={res['ROC-AUC']:.4f}")
print("=" * 70)

<a id="8"></a>
## 8. Stacked Ensemble Model

In [ ]:
# 8.1 Build a stacking classifier from the strongest base learners
estimators = [
    ("rf",  base_models["Random Forest"]),
    ("xgb", base_models["XGBoost"]),
    ("lgbm", base_models["LightGBM"]),
    ("gb",  base_models["Gradient Boosting"]),
]
stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5, n_jobs=-1, passthrough=False
)

res = evaluate_model(stack_model, X_train_scaled, y_train_sm, X_test_scaled, y_test)
results["Stacked Ensemble"] = res
all_models = {**base_models, **{f"{k} (Tuned)": v for k, v in tuned_models.items()}, "Stacked Ensemble": stack_model}

print(f"🏆 Stacked Ensemble  ->  F1={res['F1-Score']:.4f}  ROC-AUC={res['ROC-AUC']:.4f}  "
      f"Accuracy={res['Accuracy']:.4f}")

<a id="9"></a>
## 9. Model Evaluation & Comparison

In [ ]:
# 9.1 Full leaderboard
metrics_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC", "MCC", "Brier"]
leaderboard = pd.DataFrame({name: {m: res[m] for m in metrics_cols} for name, res in results.items()}).T
leaderboard = leaderboard.sort_values("F1-Score", ascending=False)

best_model_name = leaderboard.index[0]
print(f"🏆 BEST MODEL: {best_model_name}")
leaderboard.round(4).style.background_gradient(cmap="PRGn", subset=["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"])\
    .background_gradient(cmap="PRGn_r", subset=["Brier"])

In [ ]:
# 9.2 Interactive grouped bar chart — all metrics, all models
plot_metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
fig = go.Figure()
for i, name in enumerate(leaderboard.index):
    fig.add_trace(go.Bar(
        name=name, x=plot_metrics, y=leaderboard.loc[name, plot_metrics],
        marker_color=MODEL_COLORS[i % len(MODEL_COLORS)]
    ))
fig.update_layout(
    title="📈 Model Performance Comparison — All Metrics", title_x=0.5,
    barmode="group", template=PLOTLY_TEMPLATE, height=500,
    yaxis=dict(range=[0.5, 1.05], title="Score"), xaxis_title="Metric",
    legend=dict(orientation="h", y=-0.2)
)
fig.show()

In [ ]:
# 9.3 Interactive ROC curves
fig = go.Figure()
for i, name in enumerate(leaderboard.index):
    fpr, tpr, _ = roc_curve(y_test, results[name]["y_prob"])
    auc = results[name]["ROC-AUC"]
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{name} (AUC={auc:.3f})",
                              line=dict(color=MODEL_COLORS[i % len(MODEL_COLORS)], width=2)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random",
                          line=dict(color="gray", dash="dash")))
fig.update_layout(
    title="ROC Curves — All Models", title_x=0.5,
    xaxis_title="False Positive Rate", yaxis_title="True Positive Rate",
    template=PLOTLY_TEMPLATE, height=550, legend=dict(orientation="v")
)
fig.show()

In [ ]:
# 9.4 Precision-Recall curves (more informative under class imbalance)
fig = go.Figure()
for i, name in enumerate(leaderboard.index):
    prec, rec, _ = precision_recall_curve(y_test, results[name]["y_prob"])
    ap = average_precision_score(y_test, results[name]["y_prob"])
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name=f"{name} (AP={ap:.3f})",
                              line=dict(color=MODEL_COLORS[i % len(MODEL_COLORS)], width=2)))
fig.update_layout(
    title="Precision-Recall Curves — All Models", title_x=0.5,
    xaxis_title="Recall", yaxis_title="Precision",
    template=PLOTLY_TEMPLATE, height=550
)
fig.show()

In [ ]:
# 9.5 Confusion matrices — top 4 models
top4 = leaderboard.index[:4]
fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
for ax, name in zip(axes, top4):
    cm = confusion_matrix(y_test, results[name]["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"])
    disp.plot(ax=ax, cmap="Purples", colorbar=False)
    ax.set_title(name, fontweight="bold", fontsize=11)
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Confusion Matrices — Top 4 Models", fontsize=15, fontweight="bold", y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# 9.6 Calibration curves — how trustworthy are the predicted probabilities?
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfectly calibrated",
                          line=dict(color="gray", dash="dash")))
for i, name in enumerate(top4):
    frac_pos, mean_pred = calibration_curve(y_test, results[name]["y_prob"], n_bins=8, strategy="quantile")
    fig.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode="lines+markers", name=name,
                              line=dict(color=MODEL_COLORS[i % len(MODEL_COLORS)])))
fig.update_layout(
    title="Calibration Curves — Predicted Probability vs Observed Frequency", title_x=0.5,
    xaxis_title="Mean predicted probability", yaxis_title="Fraction of positives",
    template=PLOTLY_TEMPLATE, height=500
)
fig.show()

In [ ]:
# 9.7 Cross-validation F1 spread across models
fig, ax = plt.subplots(figsize=(11, 4.5))
cv_data, cv_labels = [], []
for name, model in {k: v["model"] for k, v in results.items()}.items():
    try:
        scores = cross_val_score(model, X_train_scaled, y_train_sm, cv=5, scoring="f1", n_jobs=-1)
        cv_data.append(scores)
        cv_labels.append(name)
    except Exception:
        pass

bp = ax.boxplot(cv_data, labels=cv_labels, patch_artist=True, medianprops={"color": "black", "linewidth": 2})
for patch, color in zip(bp["boxes"], MODEL_COLORS * 3):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_title("5-Fold Cross-Validation F1-Score Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("F1-Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 9.8 Learning curves for the best model — diagnose over/under-fitting
best_estimator = results[best_model_name]["model"]
train_sizes, train_scores, val_scores = learning_curve(
    best_estimator, X_train_scaled, y_train_sm, cv=5, scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1, random_state=RANDOM_STATE
)
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_sizes, y=train_scores.mean(axis=1), mode="lines+markers",
                          name="Training score", line=dict(color=PALETTE["info"])))
fig.add_trace(go.Scatter(x=train_sizes, y=val_scores.mean(axis=1), mode="lines+markers",
                          name="Validation score", line=dict(color=PALETTE["danger"])))
fig.update_layout(
    title=f"Learning Curve — {best_model_name}", title_x=0.5,
    xaxis_title="Training examples", yaxis_title="F1-Score",
    template=PLOTLY_TEMPLATE, height=450
)
fig.show()

<a id="10"></a>
## 10. Explainability — SHAP

In [ ]:
# 10.1 SHAP summary for best tree-based model (fallback to XGBoost if best isn't tree-based)
tree_models = {k: v["model"] for k, v in results.items() if "Forest" in k or "Boost" in k or "GBM" in k}
shap_model_name = best_model_name if best_model_name in tree_models else list(tree_models.keys())[0]
shap_model = tree_models[shap_model_name]

X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns)

explainer = shap.TreeExplainer(shap_model)
raw_shap = explainer.shap_values(X_test_scaled)

# Normalize across SHAP versions/output shapes to a single 2D array (n_samples, n_features)
# for the POSITIVE class (diabetes = 1).
if isinstance(raw_shap, list):
    shap_values = raw_shap[1]
    expected_value = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
elif isinstance(raw_shap, np.ndarray) and raw_shap.ndim == 3:
    # shape (n_samples, n_features, n_classes)
    shap_values = raw_shap[:, :, 1]
    ev = explainer.expected_value
    expected_value = ev[1] if isinstance(ev, (list, np.ndarray)) and np.ndim(ev) > 0 else ev
else:
    shap_values = raw_shap
    ev = explainer.expected_value
    expected_value = ev[1] if isinstance(ev, (list, np.ndarray)) and np.ndim(ev) > 0 else ev

print(f"Generating SHAP explanations using: {shap_model_name}  (shap_values shape: {shap_values.shape})")
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean |SHAP value|)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 10.2 SHAP beeswarm — direction and magnitude of feature effects
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title("SHAP Summary — Feature Effects on Diabetes Risk", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 10.3 SHAP waterfall — explain one individual high-risk prediction
idx = int(np.argmax(results[shap_model_name]["y_prob"]))
exp = shap.Explanation(
    values=shap_values[idx],
    base_values=expected_value,
    data=X_test_df.iloc[idx].values, feature_names=X.columns.tolist()
)
shap.plots.waterfall(exp, show=False)
plt.title(f"SHAP Waterfall — Highest-Risk Test Patient (#{idx})", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 10.4 Permutation importance (model-agnostic cross-check)
perm = permutation_importance(shap_model, X_test_scaled, y_test, n_repeats=20,
                                random_state=RANDOM_STATE, scoring="f1", n_jobs=-1)
perm_df = pd.DataFrame({
    "Feature": X.columns, "Importance": perm.importances_mean, "Std": perm.importances_std
}).sort_values("Importance", ascending=True)

fig = go.Figure(go.Bar(
    x=perm_df["Importance"], y=perm_df["Feature"], orientation="h",
    error_x=dict(type="data", array=perm_df["Std"]),
    marker=dict(color=perm_df["Importance"], colorscale="Viridis")
))
fig.update_layout(
    title=f"Permutation Importance — {shap_model_name}", title_x=0.5,
    xaxis_title="Mean F1 decrease when shuffled", template=PLOTLY_TEMPLATE, height=500
)
fig.show()

<a id="11"></a>
## 11. Interactive Risk Dashboard

In [ ]:
# 11.1 Build one consolidated dashboard: leaderboard + ROC + radar of best model
best_res = results[best_model_name]

radar_metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
radar_vals = [best_res[m] for m in radar_metrics]

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "polar"}, {"type": "xy"}]],
    subplot_titles=(f"Best Model Profile — {best_model_name}", "Top 5 Models by F1-Score")
)

fig.add_trace(go.Scatterpolar(
    r=radar_vals + [radar_vals[0]], theta=radar_metrics + [radar_metrics[0]],
    fill="toself", name=best_model_name, line_color=PALETTE["primary"]
), row=1, col=1)

top5 = leaderboard.head(5)
fig.add_trace(go.Bar(
    x=top5["F1-Score"], y=top5.index, orientation="h",
    marker_color=MODEL_COLORS[:5], text=top5["F1-Score"].round(3), textposition="outside"
), row=1, col=2)

fig.update_layout(
    template=PLOTLY_TEMPLATE, height=480, showlegend=False,
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title_text="🩺 Diabetes Risk Model — Executive Dashboard", title_x=0.5
)
fig.show()

In [ ]:
# 11.2 Risk-probability distribution for the test set, best model
fig = px.histogram(
    x=best_res["y_prob"], color=y_test.map({0: "No Diabetes", 1: "Diabetes"}).values,
    nbins=30, opacity=0.75, barmode="overlay",
    color_discrete_map={"No Diabetes": PALETTE["secondary"], "Diabetes": PALETTE["danger"]},
    labels={"x": "Predicted probability of diabetes", "color": "Actual outcome"},
    title=f"Predicted Risk Distribution — {best_model_name}"
)
fig.add_vline(x=0.5, line_dash="dash", line_color="black", annotation_text="Decision threshold (0.5)")
fig.update_layout(template=PLOTLY_TEMPLATE, height=450, title_x=0.5)
fig.show()

<a id="12"></a>
## 12. Live Patient Risk Predictor

In [ ]:
# 12.1 Prediction function using the full model ensemble
def predict_diabetes(pregnancies, glucose, blood_pressure, skin_thickness,
                      insulin, bmi, diabetes_pedigree, age, plot=True):
    # Predict diabetes risk for a new patient using every trained model
    # plus the stacked ensemble, and visualize the consensus.
    bmi_cat = bmi_category(bmi)
    if age <= 30:    age_grp = 0
    elif age <= 45:  age_grp = 1
    elif age <= 60:  age_grp = 2
    else:            age_grp = 3
    glc_bp_ratio   = glucose / (blood_pressure + 1)
    insulin_glc    = insulin / (glucose + 1)
    bmi_age        = bmi * age / 100

    patient = np.array([[pregnancies, glucose, blood_pressure, skin_thickness,
                          insulin, bmi, diabetes_pedigree, age,
                          bmi_cat, age_grp, glc_bp_ratio, insulin_glc, bmi_age]])
    patient_scaled = scaler.transform(patient)

    print("=" * 55)
    print("        PATIENT RISK ASSESSMENT")
    print("=" * 55)
    print(f"Glucose: {glucose} mg/dL | BP: {blood_pressure} mmHg | "
          f"BMI: {bmi} | Age: {age}")
    print("-" * 55)

    rows = []
    for name, res in results.items():
        model = res["model"]
        prob = model.predict_proba(patient_scaled)[0][1] * 100
        pred = "Diabetic" if prob >= 50 else "Healthy"
        rows.append({"Model": name, "Risk %": round(prob, 1), "Prediction": pred})

    risk_df = pd.DataFrame(rows).sort_values("Risk %", ascending=False)
    print(risk_df.to_string(index=False, formatters={"Risk %": "{:.1f}".format}))

    avg_risk = risk_df["Risk %"].mean()
    print("-" * 55)
    print(f"ENSEMBLE AVERAGE RISK: {avg_risk:.1f}%")
    if avg_risk >= 70:
        verdict = "⚠️  HIGH RISK — Immediate medical consultation recommended"
    elif avg_risk >= 40:
        verdict = "⚡ MODERATE RISK — Lifestyle changes & monitoring advised"
    else:
        verdict = "✅ LOW RISK — Maintain healthy lifestyle"
    print(verdict)
    print("=" * 55)

    if plot:
        fig = go.Figure(go.Bar(
            x=risk_df["Risk %"], y=risk_df["Model"], orientation="h",
            marker=dict(color=risk_df["Risk %"], colorscale="RdYlGn_r", cmin=0, cmax=100),
            text=risk_df["Risk %"].astype(str) + "%", textposition="outside"
        ))
        fig.add_vline(x=50, line_dash="dash", line_color="gray", annotation_text="50% threshold")
        fig.update_layout(
            title=f"Per-Model Risk Estimate — Avg = {avg_risk:.1f}%", title_x=0.5,
            template=PLOTLY_TEMPLATE, height=420, xaxis=dict(range=[0, 100], title="Risk %")
        )
        fig.show()

    return avg_risk

print("✅ Prediction function ready")

In [ ]:
# 12.2 Test Case 1 — High-Risk Patient Profile
risk1 = predict_diabetes(
    pregnancies=6, glucose=180, blood_pressure=90, skin_thickness=40,
    insulin=300, bmi=35.0, diabetes_pedigree=1.2, age=55
)

In [ ]:
# 12.3 Test Case 2 — Low-Risk Patient Profile
risk2 = predict_diabetes(
    pregnancies=1, glucose=95, blood_pressure=70, skin_thickness=22,
    insulin=80, bmi=23.5, diabetes_pedigree=0.2, age=25
)

<a id="13"></a>
## 13. Conclusion & Next Steps

In [ ]:
print("=" * 65)
print("                  PROJECT SUMMARY")
print("=" * 65)
print(f"Dataset           : Pima Indians Diabetes Dataset")
print(f"Total samples     : {len(df)}")
print(f"Features used     : {X.shape[1]} (8 original + 5 engineered)")
print(f"Train/Test split  : 80% / 20%, stratified")
print(f"Imbalance fix     : SMOTE oversampling (train only)")
print(f"Models trained    : {len(results)} (incl. tuned variants + stacked ensemble)")
print()
print("LEADERBOARD (Top 5 by F1-Score):")
print("-" * 65)
print(leaderboard[["Accuracy", "F1-Score", "ROC-AUC"]].head(5).round(4).to_string())
print("-" * 65)
print(f"\n🏆 Best model     : {best_model_name}")
print(f"   F1-Score       : {results[best_model_name]['F1-Score']:.4f}")
print(f"   ROC-AUC        : {results[best_model_name]['ROC-AUC']:.4f}")
print(f"\nKey predictive drivers (SHAP): Glucose, BMI, Age, Diabetes Pedigree Function")
print()
print("NEXT STEPS:")
print("  1. Deploy best model as REST API (FastAPI) with the stacked ensemble")
print("  2. Extend to multi-disease prediction (Heart, Kidney, Liver datasets)")
print("  3. Add a deep learning baseline (TabNet / simple MLP) for comparison")
print("  4. Build a Streamlit/Gradio UI on top of predict_diabetes()")
print("  5. Add fairness audits across age/pregnancy subgroups before clinical use")
print("  6. Integrate with EHR systems via HL7/FHIR standards")
print("=" * 65)

---
## ⚠️ Disclaimer
This notebook is for **educational and research purposes only**. It is **not a certified medical diagnostic tool**. Any risk scores produced here must not replace professional medical evaluation. Always consult a qualified healthcare provider for diagnosis and treatment decisions.

## References
1. Smith, J.W., et al. (1988). *Using the ADAP Learning Algorithm to Forecast the Onset of Diabetes Mellitus.* Proceedings of the Symposium on Computer Applications in Medical Care.
2. Lundberg, S.M. & Lee, S.I. (2017). *A Unified Approach to Interpreting Model Predictions* (SHAP). NeurIPS.
3. Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD.
4. Ke, G., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.* NeurIPS.
5. Scikit-learn Documentation — https://scikit-learn.org
6. UCI ML Repository — Pima Indians Diabetes Dataset

---
*Advanced edition — Early Disease Prediction using Machine Learning, upgraded with multi-model ensembling, hyperparameter tuning, SHAP explainability, and interactive Plotly dashboards.*